<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/19_neural_networks/19_1_Neural_Networks/19_1_2_Training_Loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neural Networks: Part 2
## The Training Loop

---

## What This Notebook Is About

In notebook 19_0_3, the training loop processed all 333 penguin samples at once in a single pass per epoch. In notebook 19_1_1, it processed all four XOR points at once. That approach is called **full-batch gradient descent**, and it works when the entire dataset fits comfortably in memory and is small enough to process quickly.

Real datasets can be millions of samples. Processing all of them before taking a single parameter update would be extremely slow. The standard solution is **mini-batch gradient descent**: split the data into small batches (typically 32–256 samples), compute the gradient on each batch, and update the parameters after every batch. Each pass through the full dataset is still one epoch, but the parameters are updated many times per epoch instead of once.

This notebook builds the complete, production-pattern training loop — the one you will use for every real problem from here on. We will apply it to the **Wisconsin Breast Cancer** dataset, which you already classified with ensemble methods in unit 18_6.

**What you will learn:**
1. How `TensorDataset` and `DataLoader` handle batching
2. The complete epoch structure: training phase followed by validation phase
3. Why `model.train()` and `model.eval()` matter
4. How to read train and validation loss curves — and what overfitting looks like
5. Why Adam is the practical default optimizer for feedforward networks

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

import seaborn as sns
sns.set_theme(style='whitegrid')
torch.manual_seed(42)

# Load the Wisconsin Breast Cancer dataset (same as unit 18_6)
data   = load_breast_cancer()
X_raw  = data.data.astype(np.float32)   # 569 samples × 30 features
y_raw  = data.target.astype(np.float32) # 0=malignant, 1=benign

print(f'Dataset: {X_raw.shape[0]} samples, {X_raw.shape[1]} features')
print(f'Classes: {dict(zip(data.target_names, np.bincount(data.target.astype(int))))}')
print()

# Three-way split: 60% train / 20% val / 20% test
# We need a separate validation set to monitor overfitting during training.
X_tmp, X_test_np, y_tmp, y_test_np = train_test_split(
    X_raw, y_raw, test_size=0.20, stratify=y_raw, random_state=42
)
X_train_np, X_val_np, y_train_np, y_val_np = train_test_split(
    X_tmp, y_tmp,  test_size=0.25, stratify=y_tmp, random_state=42  # 0.25 × 0.80 = 0.20
)

# Standardize: fit only on training data, transform all three splits
scaler     = StandardScaler()
X_train_np = scaler.fit_transform(X_train_np)
X_val_np   = scaler.transform(X_val_np)
X_test_np  = scaler.transform(X_test_np)

# Convert to PyTorch tensors
X_train = torch.tensor(X_train_np)
X_val   = torch.tensor(X_val_np)
X_test  = torch.tensor(X_test_np)
y_train = torch.tensor(y_train_np).unsqueeze(1)  # shape (n, 1) for BCELoss
y_val   = torch.tensor(y_val_np  ).unsqueeze(1)
y_test  = torch.tensor(y_test_np ).unsqueeze(1)

print(f'Train : {X_train.shape[0]} samples')
print(f'Val   : {X_val.shape[0]} samples')
print(f'Test  : {X_test.shape[0]} samples')

---

## Section 1: Batching with `DataLoader`

If you think back to the training loop in 19_0_3, it always passed the full dataset to the model: `pred = w * x + b` computed predictions for all 333 penguins at once. That is fine for small data. For larger datasets, you process one **mini-batch** at a time.

PyTorch provides two tools for this:

- **`TensorDataset`** pairs your feature and target tensors so they are always sliced together
- **`DataLoader`** wraps a dataset and yields batches of a given size; `shuffle=True` randomizes the order each epoch so the model does not learn from the sequence

With a batch size of 32 and 341 training samples, each epoch contains ceil(341/32) = 11 batches. The parameters are updated 11 times per epoch instead of once.

In [ ]:
BATCH_SIZE = 32

train_dataset = TensorDataset(X_train, y_train)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f'Training samples  : {len(train_dataset)}')
print(f'Batch size        : {BATCH_SIZE}')
print(f'Batches per epoch : {len(train_loader)}')
print()

# Each iteration of the loader yields one (X_batch, y_batch) tuple
for i, (X_batch, y_batch) in enumerate(train_loader):
    print(f'  Batch {i+1}: X shape = {X_batch.shape},  y shape = {y_batch.shape}')
    if i == 2:
        print('  ...')
        break

---

## Section 2: The Complete Training Loop

The template below is the canonical PyTorch training loop. Every real project is a variation on this structure.

Each epoch has two phases:
1. **Training phase** (`model.train()`) — iterate over batches, update parameters
2. **Validation phase** (`model.eval()` + `torch.no_grad()`) — run the full validation set without tracking gradients

The `model.train()` and `model.eval()` calls affect layers like Dropout and BatchNorm that behave differently during training vs. evaluation. We are not using those layers yet, but setting the mode correctly is a habit worth forming now.

Here is the template with every line annotated:

```
for epoch in range(n_epochs):

    model.train()                              # put model in training mode
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()                  # clear last batch's gradients
        preds = model(X_batch)                 # forward pass
        loss  = criterion(preds, y_batch)      # compute loss
        loss.backward()                        # backward pass
        optimizer.step()                       # update parameters

    model.eval()                               # switch to evaluation mode
    with torch.no_grad():                      # no gradient tracking needed
        val_loss = criterion(model(X_val), y_val)
```

In [ ]:
def build_model(hidden=(64, 32), n_features=30):
    layers = []
    in_dim = n_features
    for h in hidden:
        layers += [nn.Linear(in_dim, h), nn.ReLU()]
        in_dim = h
    layers += [nn.Linear(in_dim, 1), nn.Sigmoid()]
    return nn.Sequential(*layers)


def train_and_track(model, optimizer, n_epochs, verbose_every=50):
    criterion   = nn.BCELoss()
    train_losses, val_losses = [], []

    for epoch in range(1, n_epochs + 1):
        # --- Training phase ---
        model.train()
        epoch_loss = 0.0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            preds = model(X_batch)
            loss  = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        train_losses.append(epoch_loss / len(train_loader))

        # --- Validation phase ---
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val), y_val).item()
        val_losses.append(val_loss)

        if verbose_every and epoch % verbose_every == 0:
            print(f'  epoch {epoch:>4}  train loss: {train_losses[-1]:.4f}  val loss: {val_loss:.4f}')

    return train_losses, val_losses


torch.manual_seed(42)
model_good = build_model(hidden=(64, 32))
# weight_decay adds L2 regularization: penalizes large weights and prevents the model
# from becoming overconfident on training samples.
optimizer  = torch.optim.Adam(model_good.parameters(), lr=1e-3, weight_decay=1e-4)

print('Training a (64, 32) network for 200 epochs with Adam + weight decay:')
train_hist, val_hist = train_and_track(model_good, optimizer, n_epochs=200, verbose_every=50)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_hist, label='train loss', color='steelblue', lw=2)
ax.plot(val_hist,   label='val loss',   color='#C0392B',   lw=2, linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('Binary cross-entropy loss')
ax.set_title('Training and Validation Loss — (64, 32) Network')
ax.legend()
plt.tight_layout()
plt.show()

The training loss drops quickly as the model fits training batches with increasing confidence. The validation loss tracks the training loss early on, then gently diverges — the model gradually becomes more confident on training samples than on new data.

This mild gap is common and expected: the model has seen training data many times over, so it naturally fits it more precisely. The key question to ask is whether the validation loss eventually *rises* (a warning sign) or simply *plateaus* (acceptable). Here it rises gently from ~0.05 to ~0.07 over 200 epochs — mild overfitting in terms of loss calibration, but the classification accuracy on held-out data remains strong (as the report at the end of this notebook will show).

The `weight_decay` argument adds L2 regularization, which penalizes large weights and slows the training loss collapse. Without it the gap would be wider. What does "without it, taken to the extreme" look like?

---

## Section 3: Overfitting Is Visible in the Loss Curves

In unit 18_6, you saw overfitting as a gap between training accuracy and test accuracy as tree depth increased. The same phenomenon appears in neural networks, and the loss curves make it directly visible.

A network with far more parameters than training samples will memorize the training set. The training loss collapses to near zero. The validation loss improves early on, then starts rising — the model has stopped learning patterns and started chasing noise.

Let's make this happen deliberately with a much larger network and no regularization.

In [ ]:
torch.manual_seed(42)
model_big = build_model(hidden=(256, 256))
optimizer_big = torch.optim.Adam(model_big.parameters(), lr=1e-3)

total_params = sum(p.numel() for p in model_big.parameters())
print(f'Overfit model: {total_params:,} parameters on {X_train.shape[0]} training samples')
print(f'Parameter-to-sample ratio: {total_params / X_train.shape[0]:.1f}×  (high = prone to overfitting)')
print()
print('Training for 300 epochs:')
big_train, big_val = train_and_track(model_big, optimizer_big, n_epochs=300, verbose_every=100)

# Find where validation loss was lowest
best_epoch = int(np.argmin(big_val)) + 1

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(big_train, label='train loss', color='steelblue', lw=2)
ax.plot(big_val,   label='val loss',   color='#C0392B',   lw=2, linestyle='--')
ax.axvline(best_epoch - 1, color='#27AE60', linestyle=':', lw=1.5,
           label=f'best val epoch = {best_epoch}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title(f'Overfitting: (256, 256) Network — {total_params:,} parameters')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Train loss at epoch 300  : {big_train[-1]:.4f}')
print(f'Val   loss at epoch 300  : {big_val[-1]:.4f}')
print(f'Val   loss at best epoch : {min(big_val):.4f}  (epoch {best_epoch})')
print()
print('The gap between train and val loss is the overfitting signal.')
print('This is the same bias-variance pattern you saw in unit 18_6 decision trees,')
print('now appearing in a neural network.')

---

## Section 4: Adam vs. SGD

In notebook 19_0_3, gradient descent used a fixed learning rate for every parameter at every step. That works but requires careful tuning — too large and it diverges, too small and it crawls.

**Adam** (Adaptive Moment Estimation) improves on this in two ways:
1. It maintains a separate, adaptive learning rate for each parameter, based on how much that parameter has historically changed
2. It uses momentum — a weighted average of past gradients — to smooth out the noisy gradient estimates from mini-batches

The practical result: Adam converges faster and is less sensitive to the initial learning rate than plain SGD. It is the default optimizer for feedforward networks. `lr=1e-3` is a reasonable starting point for most problems.

Below we compare both on the same (64, 32) architecture with the same random seed.

In [ ]:
N_COMPARE = 150

torch.manual_seed(42)
m_sgd   = build_model(hidden=(64, 32))
opt_sgd = torch.optim.SGD(m_sgd.parameters(), lr=0.05, weight_decay=1e-4)

torch.manual_seed(42)
m_adam   = build_model(hidden=(64, 32))
opt_adam = torch.optim.Adam(m_adam.parameters(), lr=1e-3, weight_decay=1e-4)

sgd_train,  sgd_val  = train_and_track(m_sgd,  opt_sgd,  n_epochs=N_COMPARE, verbose_every=0)
adam_train, adam_val = train_and_track(m_adam, opt_adam, n_epochs=N_COMPARE, verbose_every=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)

for ax, t_hist, v_hist, title in [
    (axes[0], sgd_train,  sgd_val,  'SGD  (lr=0.05)'),
    (axes[1], adam_train, adam_val, 'Adam (lr=1e-3)'),
]:
    ax.plot(t_hist, color='steelblue', lw=2, label='train')
    ax.plot(v_hist, color='#C0392B',   lw=2, linestyle='--', label='val')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend()

plt.suptitle('Optimizer Comparison on Breast Cancer  (64, 32 network)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'After {N_COMPARE} epochs:')
print(f'  SGD  — train: {sgd_train[-1]:.4f}   val: {sgd_val[-1]:.4f}')
print(f'  Adam — train: {adam_train[-1]:.4f}   val: {adam_val[-1]:.4f}')
print()
print('Adam typically converges faster and to a lower loss.')
print('SGD can match it with careful learning rate tuning, but Adam removes that burden.')

---

## Section 5: Evaluating on the Test Set

The validation set was used during training to monitor overfitting and guide decisions like choosing the network size and number of epochs. The **test set** — which we have not touched yet — gives the unbiased estimate of how the model will perform on data it has never seen in any form.

After training, we use sklearn's `classification_report` for evaluation, the same function used throughout units 18_1 and 18_5. In unit 18_6, malignant recall was highlighted as the most clinically important metric — missing a cancer is worse than a false alarm. That same logic applies here.

In [ ]:
X_trainval = torch.cat([X_train, X_val])
y_trainval = torch.cat([y_train, y_val])

trainval_loader = DataLoader(
    TensorDataset(X_trainval, y_trainval),
    batch_size=BATCH_SIZE, shuffle=True
)

torch.manual_seed(42)
final_model  = build_model(hidden=(64, 32))
final_optim  = torch.optim.Adam(final_model.parameters(), lr=1e-3, weight_decay=1e-4)
final_crit   = nn.BCELoss()

for epoch in range(200):
    final_model.train()
    for X_b, y_b in trainval_loader:
        final_optim.zero_grad()
        loss = final_crit(final_model(X_b), y_b)
        loss.backward()
        final_optim.step()

final_model.eval()
with torch.no_grad():
    test_preds  = (final_model(X_test) > 0.5).long().squeeze().numpy()
    test_labels = y_test.long().squeeze().numpy()

print('Classification report on held-out test set:')
print()
print(classification_report(test_labels, test_preds, target_names=data.target_names, digits=3))
print()
print('For comparison: in unit 18_6, the best ensemble methods achieved ~97-99% accuracy')
print('on this same dataset via nested cross-validation.')
print('A simple two-layer neural network with Adam lands in a similar range.')

---

## Putting It All Together

| Concept | What it means |
|---|---|
| Mini-batch | A subset of the training data processed before each parameter update; typical size 32–128 |
| `TensorDataset` | Pairs feature and label tensors so they are sliced together |
| `DataLoader` | Yields batches from a dataset; `shuffle=True` randomizes order each epoch |
| `model.train()` | Activates training-specific behavior (Dropout, BatchNorm); call before training phase |
| `model.eval()` | Activates evaluation behavior; call before validation and inference |
| Epoch | One complete pass through the training data; parameters updated once per batch |
| Training loss | Average loss over all training batches in the epoch |
| Validation loss | Loss on the held-out validation set at epoch end — tracks generalization |
| Overfitting | Train loss falling while val loss rises; model is memorizing training noise |
| Test set | Never used during training or model selection; gives the final unbiased performance estimate |
| Adam | Adaptive optimizer: per-parameter learning rates + momentum; default for feedforward networks |

**Three-way split reasoning:** The training set updates the weights. The validation set guides decisions (network size, number of epochs, learning rate). Because it influences those decisions, it is slightly optimistic. Only the test set — unseen at every stage — gives a truly unbiased estimate. Using validation performance as your final reported result is a form of data leakage.

**What is coming next:** Notebook 19_1_3 applies the full pipeline to a multiclass problem — the CTG fetal health dataset from unit 18_5 — and introduces the softmax output layer and `CrossEntropyLoss`.